# 09 - Baseline do modelo (LightGBM): preços quebrados (P3)

Primeira versão funcional do modelo: estima o preço esperado de cada produto e alerta quando o preço real foge muito dele. Não é a versão final, é a base para as melhorias.

**Entradas:** tabelas squad1/ml/features_precos_observacoes (292.000 linhas) e squad1/ml/features_precos_sku (3.805 linhas), geradas no notebook 08.

**Saídas:** MAE do modelo contra os baselines, alertas por observação e precision, recall e taxa de alerta contra o rótulo fraco e o de apoio.

**Como rodar:** use "Executar tudo", sempre na ordem. A célula 2 instala o lightgbm (%pip) e precisa ficar logo no começo, antes de qualquer outra.

In [0]:
%pip install lightgbm

In [0]:
# ------------------------------------------------------------
# Configuração
# Importações, acesso ao lake (ADLS) e a função de leitura de tabelas Delta.
# Esta célula não lê nem grava dados.
# ------------------------------------------------------------
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F

# Lê o arquivo .env (mesma pasta do notebook), onde ficam as credenciais do ADLS.
load_dotenv(".env")

# Endereço do lake: conta de armazenamento e pasta raiz da Squad 1.
STORAGE_ACCOUNT = "internshipdatalake"
lakehouse_base = f"abfss://squad1@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Credenciais de acesso ao ADLS (autenticação OAuth com um aplicativo do Azure).
# No Serverless o spark.conf.set de credenciais é bloqueado,
# por isso elas vão em cada leitura, pelo .options(**adls_options).
adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": os.getenv("ADLS_CLIENT_ID"),
    "fs.azure.account.oauth2.client.secret": os.getenv("ADLS_CLIENT_SECRET"),
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}

def ler_delta_spark(caminho):
    """Lê uma tabela Delta do ADLS.

    caminho: endereço abfss:// da tabela (ex.: .../ml/features_precos_sku).
    Retorna um DataFrame do Spark.
    """
    return spark.read.format("delta").options(**adls_options).load(caminho)

In [0]:
# ------------------------------------------------------------
# Carga dos dados
# Lê as duas tabelas de features geradas no notebook 08 e confere
# se os totais batem com o que foi gravado.
# Não altera nada no lake: só lê.
# ------------------------------------------------------------
obs_path = f"{lakehouse_base}/ml/features_precos_observacoes"  # uma linha por observação
sku_path = f"{lakehouse_base}/ml/features_precos_sku"          # uma linha por SKU

df_obs = ler_delta_spark(obs_path)
df_sku = ler_delta_spark(sku_path)

print("features_precos_observacoes:", df_obs.count(), "linhas,", len(df_obs.columns), "colunas")
print("features_precos_sku:", df_sku.count(), "linhas,", len(df_sku.columns), "colunas")

print("\n=== Totais dos rótulos e do treino ===")
df_obs.agg(
    F.sum("rotulo_fraco").alias("rotulo_fraco"),
    F.sum("rotulo_apoio").alias("rotulo_apoio"),
    F.sum("usar_no_treino").alias("usar_no_treino"),
).show()

print("=== Linhas por conjunto ===")
df_obs.groupBy("conjunto").agg(
    F.countDistinct("sku").alias("skus"),
    F.count("*").alias("linhas"),
).show()

print("=== Colunas e tipos da tabela de observações ===")
df_obs.printSchema()

In [0]:
# ------------------------------------------------------------
# Verificações antes de treinar
# 1) regra_10: o número de categorias raiz tem que ser o esperado (12).
# 2) Nenhuma feature (nem o preço) pode ter valor nulo.
# 3) Nenhum SKU pode aparecer no treino e no teste (senão o teste não seria justo).
# Também define a lista de features usada nas próximas células.
# Não altera nada no lake: só lê.
# ------------------------------------------------------------
TOTAL_RAIZES_ESPERADO = 12          # categorias raiz 
TOTAL_SUBCATEGORIAS_ESPERADO = 123  # subcategorias vistas na EDA

# As 12 colunas de entrada do modelo. As 5 primeiras são categóricas.
FEATURES_CATEGORICAS = ["id_categoria", "id_categoria_pai", "unidade_cod", "variante_cod", "marca_cod"]
FEATURES_NUMERICAS = ["tamanho", "log_tamanho", "tamanho_relativo", "tamanho_rank_pct",
                      "marca_n_skus", "flag_generico", "flag_marca_propria"]
FEATURES = FEATURES_CATEGORICAS + FEATURES_NUMERICAS

# 1) regra_10: conta as categorias raiz (id_categoria_pai) e as subcategorias (id_categoria).
raizes = df_obs.select("id_categoria_pai").distinct().count()
subcategorias = df_obs.select("id_categoria").distinct().count()
print("Categorias raiz:", raizes, "(esperado", TOTAL_RAIZES_ESPERADO, ")")
print("Subcategorias:", subcategorias, "(esperado", TOTAL_SUBCATEGORIAS_ESPERADO, ")")
print("regra_10:", "OK" if raizes == TOTAL_RAIZES_ESPERADO else "FALHOU")

# 2) Nulos: soma quantas linhas têm valor nulo em cada coluna usada pelo modelo.
print("\n=== Nulos por coluna (tudo deve ser 0) ===")
df_obs.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in FEATURES + ["preco"]]
).show(vertical=True)

# 3) SKUs presentes ao mesmo tempo no treino e no teste.
skus_treino = df_obs.filter(F.col("conjunto") == "treino").select("sku").distinct()
skus_teste = df_obs.filter(F.col("conjunto") == "teste").select("sku").distinct()
print("SKUs em treino e teste ao mesmo tempo:", skus_treino.intersect(skus_teste).count(), "(esperado 0)")

In [0]:
# ------------------------------------------------------------
# Baselines no conjunto de teste
# Recalcula os baselines da EDA de forma justa:
#   - as medianas são calculadas só com as linhas de TREINO;
#   - o erro (MAE) é medido só nas linhas de TESTE.
# Também traz os dados para o pandas, que o LightGBM usa nas próximas células.
# Não altera nada no lake: só lê.
# ------------------------------------------------------------
import numpy as np
import pandas as pd

pdf = df_obs.toPandas()   # 292.000 linhas cabem tranquilamente na memória

# Treino: folds 1 a 4, só linhas "normais" (usar_no_treino == 1).
# Teste: fold 0, só linhas "normais" (para o MAE não ser puxado pelos erros gigantes).
treino = pdf[(pdf["conjunto"] == "treino") & (pdf["usar_no_treino"] == 1)].copy()
teste = pdf[(pdf["conjunto"] == "teste") & (pdf["usar_no_treino"] == 1)].copy()
print("Linhas de treino:", len(treino), "| Linhas de teste:", len(teste))

def mae(real, previsto):
    """Erro absoluto médio (em R$): média do tamanho do erro, sem sinal."""
    return float(np.mean(np.abs(real - previsto)))

# Baseline 1: mediana global do treino (o mesmo valor para todo produto).
mediana_global = treino["preco"].median()
pred_global = pd.Series(mediana_global, index=teste.index)

# Baseline 2: mediana do treino por subcategoria.
# Se uma subcategoria não existir no treino, usa a mediana global.
med_subcat = treino.groupby("id_categoria")["preco"].median()
pred_subcat = teste["id_categoria"].map(med_subcat).fillna(mediana_global)

# Baseline 3: mediana do treino por subcategoria + tamanho.
# Se a combinação não existir no treino, usa a mediana da subcategoria.
med_subcat_tam = treino.groupby(["id_categoria", "tamanho"])["preco"].median()
chave = pd.MultiIndex.from_frame(teste[["id_categoria", "tamanho"]])
pred_subcat_tam = pd.Series(med_subcat_tam.reindex(chave).values, index=teste.index)
sem_combinacao = int(pred_subcat_tam.isna().sum())   # linhas que precisaram do plano B
pred_subcat_tam = pred_subcat_tam.fillna(pred_subcat)

# Piso: a mediana do próprio SKU (coluna mediana_sku).
# Usa informação do SKU de teste, que o modelo real não terá: serve só como
# referência de "o melhor que conhecer o produto permite".
pred_piso = teste["mediana_sku"]

resultados = pd.DataFrame({
    "baseline": ["mediana global", "subcategoria", "subcategoria + tamanho", "mediana do SKU (piso)"],
    "MAE_R$": [mae(teste["preco"], p) for p in [pred_global, pred_subcat, pred_subcat_tam, pred_piso]],
})
print("\n=== MAE no teste (R$) ===")
print(resultados.round(2).to_string(index=False))
print("\nMediana global do treino: R$", round(mediana_global, 2))
print("Linhas do teste sem combinação subcategoria+tamanho no treino:", sem_combinacao)

In [0]:
# ------------------------------------------------------------
# Verificação da biblioteca LightGBM
# Confere se o lightgbm está instalado no ambiente e mostra a versão.
# Não lê nem grava dados.
# ------------------------------------------------------------
import lightgbm as lgb
print("LightGBM versão:", lgb.__version__)

In [0]:
# ------------------------------------------------------------
# Treino do LightGBM com parada antecipada
# Descobre quantas árvores usar. O treino usa os folds 1 a 3 e o fold 4
# serve de validação (SKUs que o modelo não viu), para parar quando o erro
# de validação deixar de melhorar. O teste (fold 0) NÃO é usado aqui.
# Não altera nada no lake.
# ------------------------------------------------------------
import lightgbm as lgb

# Separa treino e validação pelo fold (a divisão por SKU já vem do notebook 08).
parte_treino = treino[treino["fold"] != 4]
parte_valid = treino[treino["fold"] == 4]
print("Linhas de treino:", len(parte_treino), "| Linhas de validação:", len(parte_valid))

modelo = lgb.LGBMRegressor(
    objective="regression_l1",   # perda robusta: o modelo estima a mediana, não é puxado por erros gigantes
    n_estimators=2000,           # máximo de árvores (a parada antecipada corta antes disso)
    learning_rate=0.05,          # quanto cada árvore corrige da anterior (pequeno = mais cuidadoso)
    num_leaves=31,               # tamanho máximo de cada árvore
    min_child_samples=100,       # cada folha precisa de pelo menos 100 linhas (evita decorar SKUs)
    random_state=42,             # resultado repetível
    verbose=-1,                  # silencia mensagens internas
)

modelo.fit(
    parte_treino[FEATURES], parte_treino["preco"],
    eval_set=[(parte_valid[FEATURES], parte_valid["preco"])],
    eval_metric="l1",                         # l1 = MAE
    categorical_feature=FEATURES_CATEGORICAS, # as 5 colunas categóricas
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),  # para se não melhorar em 50 árvores seguidas
        lgb.log_evaluation(period=100),          # imprime o erro a cada 100 árvores
    ],
)

melhor_n = modelo.best_iteration_
print("\nMelhor número de árvores:", melhor_n)
pred_valid = modelo.predict(parte_valid[FEATURES], num_iteration=melhor_n)
print("MAE na validação (fold 4): R$", round(mae(parte_valid["preco"], pred_valid), 2))

In [0]:
# ------------------------------------------------------------
# Modelo final e comparação com os baselines no teste
# Treina o LightGBM com todo o treino (folds 1 a 4) usando o número de árvores
# encontrado na parada antecipada, prevê o preço dos SKUs de teste (fold 0)
# e compara o MAE com os baselines.
# Não altera nada no lake.
# ------------------------------------------------------------
modelo_final = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=melhor_n,       # número de árvores achado na célula anterior
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=100,
    random_state=42,
    verbose=-1,
)
modelo_final.fit(
    treino[FEATURES], treino["preco"],
    categorical_feature=FEATURES_CATEGORICAS,
)

# Preço esperado de cada observação do teste (guardado para calcular os alertas depois).
teste["previsto"] = modelo_final.predict(teste[FEATURES])
mae_lgbm = mae(teste["preco"], teste["previsto"])

# Tabela de comparação: os baselines da célula 6 mais o LightGBM.
comparacao = pd.concat(
    [resultados.rename(columns={"baseline": "modelo"}),
     pd.DataFrame({"modelo": ["LightGBM"], "MAE_R$": [mae_lgbm]})],
    ignore_index=True,
)
print("Linhas de treino:", len(treino), "| Árvores:", melhor_n)
print("\n=== MAE no teste (R$) ===")
print(comparacao.round(2).to_string(index=False))

# Ganho do LightGBM sobre o baseline "subcategoria + tamanho".
base = float(resultados.loc[resultados["baseline"] == "subcategoria + tamanho", "MAE_R$"].iloc[0])
print("\nGanho sobre 'subcategoria + tamanho':", round((base - mae_lgbm) / base * 100, 1), "%")

# Quais colunas o modelo mais usou (número de vezes que cada uma foi usada nas árvores).
importancia = pd.Series(modelo_final.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\n=== Uso de cada feature nas árvores ===")
print(importancia.to_string())

In [0]:
# ------------------------------------------------------------
# Comparação de conjuntos de features
# Para cada conjunto: parada antecipada no fold 4 (acha o nº de árvores),
# refaz o treino com todo o treino e mede o MAE no teste.
# A escolha do conjunto deve usar o MAE de validação; o de teste só confirma.
# Não altera nada no lake.
# ------------------------------------------------------------
def treinar_e_avaliar(features, categoricas, n_max=2000):
    """Treina com parada antecipada, refaz com todo o treino e mede no teste.

    features: lista de colunas de entrada.
    categoricas: quais dessas colunas são categóricas.
    Retorna (nº de árvores, MAE na validação, MAE no teste).
    """
    parte_treino = treino[treino["fold"] != 4]
    parte_valid = treino[treino["fold"] == 4]
    params = dict(objective="regression_l1", learning_rate=0.05, num_leaves=31,
                  min_child_samples=100, random_state=42, verbose=-1)

    # Etapa 1: acha quantas árvores usar (o fold 4 serve de validação).
    m = lgb.LGBMRegressor(n_estimators=n_max, **params)
    m.fit(parte_treino[features], parte_treino["preco"],
          eval_set=[(parte_valid[features], parte_valid["preco"])], eval_metric="l1",
          categorical_feature=categoricas,
          callbacks=[lgb.early_stopping(50, verbose=False)])
    n = m.best_iteration_
    mae_val = mae(parte_valid["preco"], m.predict(parte_valid[features], num_iteration=n))

    # Etapa 2: refaz o treino com todo o treino e mede no teste.
    final = lgb.LGBMRegressor(n_estimators=n, **params)
    final.fit(treino[features], treino["preco"], categorical_feature=categoricas)
    mae_teste = mae(teste["preco"], final.predict(teste[features]))
    return n, mae_val, mae_teste

# Colunas ligadas à marca e à variante, que vamos tirando.
cols_marca = ["marca_cod", "marca_n_skus", "flag_generico", "flag_marca_propria"]
cenarios = {
    "todas (12)": FEATURES,
    "sem marca (8)": [f for f in FEATURES if f not in cols_marca],
    "sem marca e sem variante (7)": [f for f in FEATURES if f not in cols_marca + ["variante_cod"]],
}

linhas = []
for nome, feats in cenarios.items():
    cats = [f for f in FEATURES_CATEGORICAS if f in feats]
    n, mae_val, mae_teste = treinar_e_avaliar(feats, cats)
    linhas.append({"conjunto": nome, "arvores": n,
                   "MAE_validacao": mae_val, "MAE_teste": mae_teste})

print("=== Comparação de conjuntos de features (R$) ===")
print(pd.DataFrame(linhas).round(2).to_string(index=False))
print("\nReferência no teste: subcategoria + tamanho = R$ 9.44 | piso (mediana do SKU) = R$ 9.23")

In [0]:
# ------------------------------------------------------------
# Modelo base e preço esperado em todo o teste
# Treina o LightGBM com o conjunto de features escolhido (7 colunas, sem marca
# e sem variante) e prevê o preço esperado das 58.546 linhas do teste, incluindo
# as linhas quebradas. Depois compara preço real / preço esperado por grupo.
# Não altera nada no lake.
# ------------------------------------------------------------
FEATURES_BASE = [f for f in FEATURES if f not in cols_marca + ["variante_cod"]]
CATEGORICAS_BASE = [f for f in FEATURES_CATEGORICAS if f in FEATURES_BASE]
N_ARVORES_BASE = 68   # achado na parada antecipada da célula anterior

modelo_base = lgb.LGBMRegressor(
    objective="regression_l1", n_estimators=N_ARVORES_BASE, learning_rate=0.05,
    num_leaves=31, min_child_samples=100, random_state=42, verbose=-1,
)
modelo_base.fit(treino[FEATURES_BASE], treino["preco"], categorical_feature=CATEGORICAS_BASE)
print("Features do modelo base:", FEATURES_BASE)

# Preço esperado de TODAS as linhas do teste (normais e quebradas).
teste_todas = pdf[pdf["conjunto"] == "teste"].copy()
teste_todas["previsto"] = modelo_base.predict(teste_todas[FEATURES_BASE])
teste_todas["razao"] = teste_todas["preco"] / teste_todas["previsto"]   # 1,0 = preço igual ao esperado

print("Linhas do teste:", len(teste_todas), "| menor preço esperado: R$", round(teste_todas["previsto"].min(), 2))

# MAE só nas linhas normais: deve repetir o valor da célula anterior (perto de 9,45).
normais = teste_todas[teste_todas["usar_no_treino"] == 1]
print("MAE nas linhas normais: R$", round(mae(normais["preco"], normais["previsto"]), 2))

# Como a razão preço real / esperado se comporta em cada grupo.
teste_todas["grupo"] = np.select(
    [teste_todas["rotulo_fraco"] == 1, teste_todas["rotulo_apoio"] == 1],
    ["erro (rótulo fraco)", "só rótulo de apoio"],
    default="normal",
)
resumo = teste_todas.groupby("grupo")["razao"].describe(percentiles=[0.05, 0.5, 0.95])
print("\n=== Razão preço real / esperado por grupo ===")
print(resumo[["count", "min", "5%", "50%", "95%", "max"]].round(2).to_string())

In [0]:
# ------------------------------------------------------------
# Alerta por resíduo e MAD + métricas
# Score = log(preço real / preço esperado). A "bagunça típica" (MAD) de cada
# categoria raiz vem só das linhas normais do treino. Alerta quando |z| > k.
# Calcula precision, recall e taxa de alerta contra o rótulo fraco e o de apoio,
# para vários valores de k, no treino e no teste.
# ATENÇÃO: no treino, o preço esperado vem de um modelo que já viu essas linhas,
# então a tabela do treino é só um guia para escolher k, não um resultado final.
# Não altera nada no lake.
# ------------------------------------------------------------
def calcular_score(df):
    """Score = log(preço real / preço esperado). Preço <= 0 vira 0,01 para o log existir."""
    return np.log(np.maximum(df["preco"], 0.01) / df["previsto"])

# Preço esperado e score das linhas normais do treino (base para o centro e a MAD).
treino["previsto"] = modelo_base.predict(treino[FEATURES_BASE])
treino["score"] = calcular_score(treino)

# Centro e MAD globais (plano B) e por categoria raiz.
centro_global = treino["score"].median()
mad_global = (treino["score"] - centro_global).abs().median()
por_raiz = treino.groupby("id_categoria_pai")["score"]
centro_raiz = por_raiz.median()
mad_raiz = por_raiz.apply(lambda s: (s - s.median()).abs().median())
print("Centro global:", round(centro_global, 3), "| MAD global:", round(mad_global, 3))
print("MAD por categoria raiz: de", round(mad_raiz.min(), 3), "a", round(mad_raiz.max(), 3))

def calcular_z(df):
    """Distância do score ao centro do grupo, em unidades de 'bagunça típica' (MAD)."""
    centro = df["id_categoria_pai"].map(centro_raiz).fillna(centro_global)
    mad = df["id_categoria_pai"].map(mad_raiz).fillna(mad_global)
    return (calcular_score(df) - centro) / (1.4826 * mad)

# Todas as linhas do treino (normais e quebradas) e do teste, com o z de cada uma.
treino_todas = pdf[pdf["conjunto"] == "treino"].copy()
treino_todas["previsto"] = modelo_base.predict(treino_todas[FEATURES_BASE])
treino_todas["z"] = calcular_z(treino_todas)
teste_todas["z"] = calcular_z(teste_todas)

def avaliar_alertas(df, k):
    """Precision, recall e taxa de alerta (em %) para o limite k."""
    alerta = df["z"].abs() > k
    n_alertas = int(alerta.sum())
    saida = {"k": k, "taxa_alerta_%": alerta.mean() * 100}
    for rotulo in ["fraco", "apoio"]:
        real = df["rotulo_" + rotulo] == 1
        acertos = int((alerta & real).sum())
        saida["precision_" + rotulo] = acertos / n_alertas * 100 if n_alertas > 0 else np.nan
        saida["recall_" + rotulo] = acertos / int(real.sum()) * 100
    return saida

ks = [3, 4, 5, 6, 8, 10]
for nome, df in [("TREINO (só guia para escolher k)", treino_todas), ("TESTE", teste_todas)]:
    tabela = pd.DataFrame([avaliar_alertas(df, k) for k in ks])
    print("\n===", nome, "| erros reais: fraco", int(df["rotulo_fraco"].sum()),
          "| apoio", int(df["rotulo_apoio"].sum()), "de", len(df), "linhas ===")
    print(tabela.round(1).to_string(index=False))

In [0]:
# ------------------------------------------------------------
# Resumo final do baseline (k = 5)
# Fixa o limite do alerta, mostra precision, recall e taxa de alerta no teste
# e separa os alertas por tipo.
# Não altera nada no lake.
# ------------------------------------------------------------
K_ALERTA = 5   # escolhido olhando o treino: está no meio do trecho estável (k de 4 a 5)

teste_todas["alerta"] = teste_todas["z"].abs() > K_ALERTA
teste_todas["tipo_alerta"] = np.select(
    [~teste_todas["alerta"], teste_todas["preco"] <= 0, teste_todas["z"] > 0],
    ["sem alerta", "preço zero ou negativo", "preço muito acima do esperado"],
    default="preço muito abaixo do esperado (positivo)",
)

print("=== Alertas no teste por tipo (k =", K_ALERTA, ") ===")
print(teste_todas["tipo_alerta"].value_counts().to_string())

print("\n=== Métricas no teste (%) ===")
print(pd.DataFrame([avaliar_alertas(teste_todas, K_ALERTA)]).round(1).to_string(index=False))

print("\nTotal de alertas:", int(teste_todas["alerta"].sum()), "de", len(teste_todas), "linhas")
print("MAE nas linhas normais: R$", round(mae(normais["preco"], normais["previsto"]), 2))

## Conclusões do baseline

**O que foi feito:** LightGBM de regressão (perda L1, 68 árvores) com 7 colunas de entrada, treinado nos folds 1 a 4 (228.237 linhas) e testado no fold 0 (765 SKUs que o modelo não viu; 57.212 linhas normais e 58.546 no total). O número de árvores veio da parada antecipada no fold 4.

**1. MAE no teste (R$):** mediana global 15,22; subcategoria 10,32; subcategoria + tamanho 9,44; mediana do SKU (piso) 9,23; LightGBM 9,45.

**2. Escolha das colunas:** com as 12 colunas o MAE foi 9,53; sem marca (8 colunas) 9,50; sem marca e sem variante (7 colunas) 9,45, com o melhor MAE de validação (9,48). Marca e variante não ajudam: o modelo as usava para decorar SKUs do treino.

**3. O LightGBM empata com a mediana por subcategoria + tamanho.** Cada observação tem ruído independente de cerca de 35%, e o piso de 9,23 usa a mediana do próprio SKU do teste, informação que um SKU novo não tem. O valor do modelo está no alerta, não no MAE.

**4. Alerta:** score = log(preço real / preço esperado), com preço menor ou igual a zero tratado como R$ 0,01. A bagunça típica (MAD) vem das linhas normais do treino, por categoria raiz. Alerta quando |z| > 5, com z = (score - centro) / (1,4826 x MAD). O limite 5 foi escolhido olhando o treino (trecho estável entre 4 e 5).

**5. Resultado do alerta no teste:** 1.334 alertas em 58.546 linhas (2,3%), sendo 900 de preço zero ou negativo e 434 de preço muito acima do esperado. Contra o rótulo de apoio: precision 100% e recall 100%. Contra o rótulo fraco: precision 68,8% e recall 100%; as 416 linhas só do rótulo de apoio são erros reais que o rótulo fraco não cobre, por isso contam como falso alarme.

**6. Limitações:** os rótulos vêm de regras simples e os erros são extremos (razão menor ou igual a 0 ou maior ou igual a 14, contra 0,23 a 2,09 no normal), então o resultado perfeito confirma o funcionamento, mas não prova que o modelo pegue erros moderados, como um preço 2 ou 3 vezes fora, que não têm rótulo. A MAD vem de previsões feitas em linhas que o modelo já viu, o que a deixa um pouco otimista.

**7. Próximos passos:** medir o alerta em erros moderados injetados de forma sintética; calcular a MAD por subcategoria; ajustar os hiperparâmetros; testar novas colunas.